In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print('All imports successful.')

All imports successful.


In [3]:
diabetes_path = '../data/diabetes/diabetes.csv'
diabetes_df = pd.read_csv(diabetes_path)

print('Shape:', diabetes_df.shape)
diabetes_df.head()

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
# Replace biologically invalid zeros with NaN
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_with_invalid_zeros:
    diabetes_df[col] = diabetes_df[col].replace(0, np.nan)

print('Missing values after zero-replacement:')
print(diabetes_df.isnull().sum())

Missing values after zero-replacement:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [5]:
# Impute with median (robust to outliers in medical data)
for col in cols_with_invalid_zeros:
    diabetes_df[col] = diabetes_df[col].fillna(diabetes_df[col].median())

print('Missing values after imputation:')
print(diabetes_df.isnull().sum().sum())

Missing values after imputation:
0


In [6]:
#Check duplicate rows
print("Duplicate rows:", diabetes_df.duplicated().sum())

# Remove duplicates if any
diabetes_df = diabetes_df.drop_duplicates()

print("Shape after removing duplicates:", diabetes_df.shape)

Duplicate rows: 0
Shape after removing duplicates: (768, 9)


In [7]:
# Separate features and target
X = diabetes_df.drop("Outcome", axis=1)
y = diabetes_df["Outcome"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (768, 8)
Target shape: (768,)


In [8]:
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Categorical columns: []


In [9]:
# Impute missing values
from sklearn.impute import SimpleImputer

# Median imputation for numerical columns
num_imputer = SimpleImputer(strategy="median")
X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])

print("Missing values after imputation:")
print(X.isnull().sum())

Missing values after imputation:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [10]:
# Scale numerical features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("Numerical features scaled successfully.")

Numerical features scaled successfully.


In [11]:
#  Combine processed features with target
processed_diabetes_df = pd.concat([X, y], axis=1)

print("Processed diabetes dataset shape:", processed_diabetes_df.shape)
processed_diabetes_df.head()

Processed diabetes dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.639947,0.866045,-0.031990,0.670643,-0.181541,0.166619,0.468492,1.425995,1
1,-0.844885,-1.205066,-0.528319,-0.012301,-0.181541,-0.852200,-0.365061,-0.190672,0
2,1.233880,2.016662,-0.693761,-0.012301,-0.181541,-1.332500,0.604397,-0.105584,1
3,-0.844885,-1.073567,-0.528319,-0.695245,-0.540642,-0.633881,-0.920763,-1.041549,0
4,-1.141852,0.504422,-2.679076,0.670643,0.316566,1.549303,5.484909,-0.020496,1


In [12]:
# Final check
print("Final missing values:")
print(processed_diabetes_df.isnull().sum())

print("\nFinal data types:")
print(processed_diabetes_df.dtypes)

Final missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Final data types:
Pregnancies                 float64
Glucose                     float64
BloodPressure               float64
SkinThickness               float64
Insulin                     float64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                         float64
Outcome                       int64
dtype: object


In [13]:
# Save cleaned dataset
processed_diabetes_df.to_csv("../data/diabetes/diabetes_cleaned.csv", index=False)

print("Preprocessed diabetes dataset saved successfully.")

Preprocessed diabetes dataset saved successfully.


In [14]:
# Outlier clipping (IQR method) 
# Insulin and SkinThickness are heavily skewed in this dataset.
# We clip at 1.5×IQR fence to reduce the effect of extreme values
# without removing rows (important for a small 768-row dataset).

cols_to_clip = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in cols_to_clip:
    Q1 = diabetes_df[col].quantile(0.25)
    Q3 = diabetes_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    diabetes_df[col] = diabetes_df[col].clip(lower=lower, upper=upper)

print('Outlier clipping done.')
diabetes_df[cols_to_clip].describe()

Outlier clipping done.


,Glucose,BloodPressure,SkinThickness,Insulin,BMI
count,768.000000,768.000000,768.000000,768.000000,768.000000
mean,121.656250,72.358073,28.866536,124.691081,32.393359
std,30.438286,11.697097,7.442353,7.913595,6.667471
min,44.000000,40.000000,14.500000,112.875000,18.200000
25%,99.750000,64.000000,25.000000,121.500000,27.500000
50%,117.000000,72.000000,29.000000,125.000000,32.300000
75%,140.250000,80.000000,32.000000,127.250000,36.600000
max,199.000000,104.000000,42.500000,135.875000,50.250000


In [15]:
# Separate features and target 

X = diabetes_df.drop(columns=['Outcome'])
y = diabetes_df['Outcome']

print('Feature matrix shape:', X.shape)
print('Target distribution:')
print(y.value_counts())
print(f'\nClass imbalance ratio: {y.value_counts()[0]}:{y.value_counts()[1]}')

Feature matrix shape: (768, 8)
Target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Class imbalance ratio: 500:268


In [16]:
# Train / test split 
# stratify=y ensures both splits preserve the original class ratio.
# random_state=42 makes results reproducible.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print('\nTrain class distribution:')
print(y_train.value_counts())
print('\nTest class distribution:')
print(y_test.value_counts())

Training set : 614 samples
Test set     : 154 samples

Train class distribution:
Outcome
0    400
1    214
Name: count, dtype: int64

Test class distribution:
Outcome
0    100
1     54
Name: count, dtype: int64


In [17]:
# Feature scaling 
# CRITICAL: fit the scaler on X_train ONLY, then transform both sets.
# Fitting on the full dataset would leak test-set statistics into training.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # transform only on test

# Convert back to DataFrames to keep column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('Scaling done.')
print('\nTrain scaled stats (should be ~mean=0, std=1):')
print(X_train_scaled.describe().loc[['mean', 'std']].round(3))

Scaling done.

Train scaled stats (should be ~mean=0, std=1):
      Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin    BMI  \
mean       -0.000   -0.000         -0.000          0.000    0.000 -0.000   
std         1.001    1.001          1.001          1.001    1.001  1.001   

      DiabetesPedigreeFunction    Age  
mean                    -0.000 -0.000  
std                      1.001  1.001  
